In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.quantization
from transformers import AutoModelForAudioClassification,ASTFeatureExtractor



In [ ]:
# 1. Load your fine-tuned model
model_path = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
model = AutoModelForAudioClassification.from_pretrained(model_path)
model.eval()


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification(
  (audio_spectrogram_transformer): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=T

In [ ]:

# 2. Assign Quantization Configuration
# 'fbgemm' is for x86 CPUs (like your Dell Inspiron); 'qnnpack' is for ARM (mobile).
model.qconfig = torch.quantization.get_default_qconfig('fbgemm')

# for mobile devices
# model.qconfig = torch.quantization.get_default_qconfig('qnnpack')



In [ ]:
# 3. Prepare the model for quantization
# This inserts "observers" that watch the data during calibration
model_prepared = torch.quantization.prepare(model)


/tmp/ipykernel_9907/1347395045.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = torch.quantization.prepare(model)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of 

In [ ]:
feature_extractor = ASTFeatureExtractor.from_pretrained(model_path)

In [5]:
from datasets import load_from_disk
dataset_path = "/content/drive/MyDrive/Urban8KAudioFiles/processed_train"
test_dataset = load_from_disk(dataset_path)

In [ ]:
print(test_dataset.column_names)

['input_values', 'labels']


In [ ]:
import torch

# 3. Calibration Step
print("Calibrating for Quantization using 'input_values'...")
# model_prepared.to('cpu') # Ensure model is on CPU for quantization
model_prepared.eval()

with torch.no_grad():
    for i in range(20):
        # 1. Get the pre-processed spectrogram patches
        # We wrap it in a list [ ] and convert to tensor to create the batch dimension
        feat = torch.tensor([test_dataset[i]["input_values"]])

        # 2. Prepare the dictionary for the model
        inputs = {"input_values": feat}

        # 3. Forward pass for calibration
        model_prepared(**inputs)

print("Calibration successful! The observers have recorded the dynamic range.")

Calibrating for Quantization using 'input_values'...
Calibration successful! The observers have recorded the dynamic range.


In [ ]:
# 5. Convert to Quantized Model
model_int8 = torch.quantization.convert(model_prepared)



/tmp/ipykernel_9907/2398995763.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.convert(model_prepared)


In [ ]:
# 6. Save the Quantized Model
torch.save(model_int8.state_dict(), f"{model_path}/ast_model_int8_v2.pth")
print("Quantization Complete! Model size reduced by ~4x.")

Quantization Complete! Model size reduced by ~4x.


# Testing Quantization model

In [ ]:
# Force the model to the CPU
model_int8.to('cpu')

def evaluate_quantized_model(model, dataset):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i in range(len(dataset)):
            # Ensure the input tensor is explicitly on the CPU
            inputs = torch.tensor([dataset[i]["input_values"]]).to('cpu')

            logits = model(inputs).logits
            prediction = torch.argmax(logits, dim=-1).item()

            all_preds.append(prediction)
            all_labels.append(dataset[i]["labels"])
    # ... rest of your accuracy calculation

In [ ]:
evaluate_quantized_model(model_int8,test_dataset)

In [ ]:
import torch
import torchaudio
import time
from transformers import ASTFeatureExtractor, AutoModelForAudioClassification

# 1. Paths
model_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
test_audio_path = "test_sound.wav" # Replace with your file

# 2. Load the base model directly to CPU
print("Loading model structure...")
model = AutoModelForAudioClassification.from_pretrained(model_dir)
model.to('cpu')
model.eval()

# 3. Apply Robust Dynamic Quantization
# We target the Linear layers (nn.Linear) which dominate the Transformer attention blocks
print("Applying Dynamic Quantization to Transformer Linear layers...")
model_int8 = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear}, # Target layers
    dtype=torch.qint8  # Quantize weights to INT8
)

# 4. Process the Audio File
feature_extractor = ASTFeatureExtractor.from_pretrained(model_dir)

# Load and resample to 16kHz
audio, sr = torchaudio.load(test_audio_path)
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    audio = resampler(audio)

# Convert to Mel Spectrogram patches
inputs = feature_extractor(audio.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to('cpu') for k, v in inputs.items()}

# 5. Run Error-Free Inference
print("Running inference on CPU...")
start_time = time.time()
with torch.no_grad():
    outputs = model_int8(inputs['input_values'])
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()
latency = (time.time() - start_time) * 1000
urban8KSounds = {
    0: "air_conditioner",
    1: "car_horn",
    2: "children_playing",
    3: "dog_bark",
    4: "drilling",
    5: "engine_idling",
    6: "gun_shot",
    7: "jackhammer",
    8: "siren",
    9: "street_music"
  }
# 6. Map to Class Name
class_name = urban8KSounds.get(prediction, f"Unknown_Label_{prediction}")
print("\n--- Inference Results ---")
print(f"Prediction: {class_name}")
print(f"Inference Latency: {latency:.2f} ms")

Loading model structure...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Applying Dynamic Quantization to Transformer Linear layers...


/tmp/ipykernel_4574/2693534951.py:19: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


Running inference on CPU...

--- Inference Results ---
Prediction: air_conditioner
Inference Latency: 3552.02 ms


In [ ]:
import torch
from transformers import AutoModelForAudioClassification

model_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
quantized_model_path = f"{model_dir}/ast_dynamic_int8.pth"
test_audio_path = "test1.wav" # Replace with your file
print("1. Rebuilding base model architecture...")
# Load the unquantized architecture shell directly to CPU
base_model = AutoModelForAudioClassification.from_pretrained(model_dir)
base_model.to('cpu')

print("2. Instantiating dynamic quantization layers...")
# Dynamically quantize the shell to match the exact layer transformation of the saved weights
loaded_model_int8 = torch.quantization.quantize_dynamic(
    base_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("3. Loading quantized weights into the architecture...")
# Inject the saved INT8 weights into our structured model
loaded_model_int8.load_state_dict(torch.load(quantized_model_path, map_location='cpu'))
loaded_model_int8.eval()

print("Model is fully loaded and ready for inference!")

# 4. Process the Audio File
feature_extractor = ASTFeatureExtractor.from_pretrained(model_dir)

# Load and resample to 16kHz
audio, sr = torchaudio.load(test_audio_path)
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    audio = resampler(audio)

# Convert to Mel Spectrogram patches
inputs = feature_extractor(audio.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to('cpu') for k, v in inputs.items()}

# 5. Run Error-Free Inference
print("Running inference on CPU...")
start_time = time.time()
with torch.no_grad():
    outputs = loaded_model_int8(inputs['input_values'])
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()
latency = (time.time() - start_time) * 1000
urban8KSounds = {
    0: "air_conditioner",
    1: "car_horn",
    2: "children_playing",
    3: "dog_bark",
    4: "drilling",
    5: "engine_idling",
    6: "gun_shot",
    7: "jackhammer",
    8: "siren",
    9: "street_music"
  }
# 6. Map to Class Name
class_name = urban8KSounds.get(prediction, f"Unknown_Label_{prediction}")
print("\n--- Inference Results ---")
print(f"Prediction: {class_name}")
print(f"Inference Latency: {latency:.2f} ms")

1. Rebuilding base model architecture...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

2. Instantiating dynamic quantization layers...


/tmp/ipykernel_4574/2486576046.py:14: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  loaded_model_int8 = torch.quantization.quantize_dynamic(


3. Loading quantized weights into the architecture...
Model is fully loaded and ready for inference!
Running inference on CPU...

--- Inference Results ---
Prediction: children_playing
Inference Latency: 4414.01 ms


# Saving Model

In [ ]:
import os

# Define your save path
save_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
quantized_model_path = os.path.join(save_dir, "ast_dynamic_int8.pth")

# Save only the weights (state_dict) of the quantized model
torch.save(model_int8.state_dict(), quantized_model_path)
print(f"Quantized model successfully saved to: {quantized_model_path}")

Quantized model successfully saved to: /content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model/ast_dynamic_int8.pth


In [ ]:
original_size = os.path.getsize(f"{model_dir}/model.safetensors") / (1024 * 1024)
quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024)

print(f"Original Model Size: {original_size:.2f} MB")
print(f"Quantized Model Size: {quantized_size:.2f} MB")
print(f"Storage Reduction: {((original_size - quantized_size) / original_size) * 100:.2f}%")

Original Model Size: 328.84 MB
Quantized Model Size: 85.94 MB
Storage Reduction: 73.87%


In [ ]:
from transformers import AutoModelForAudioClassification

model_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
fine_tuned_model = AutoModelForAudioClassification.from_pretrained(model_dir)

# 1. Calculate Total and Trainable Parameters
total_params = sum(p.numel() for p in fine_tuned_model.parameters())
trainable_params = sum(p.numel() for p in fine_tuned_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters:    {frozen_params:,}")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Total Parameters:     86,196,490
Trainable Parameters: 86,196,490
Frozen Parameters:    0


In [ ]:
print(f"{'Layer Name':<60} | {'Parameters':<15}")
print("-" * 80)
for name, param in fine_tuned_model.named_parameters():
    print(f"{name:<60} | {param.numel():<15,}")

Layer Name                                                   | Parameters     
--------------------------------------------------------------------------------
audio_spectrogram_transformer.embeddings.cls_token           | 768            
audio_spectrogram_transformer.embeddings.distillation_token  | 768            
audio_spectrogram_transformer.embeddings.position_embeddings | 932,352        
audio_spectrogram_transformer.embeddings.patch_embeddings.projection.weight | 196,608        
audio_spectrogram_transformer.embeddings.patch_embeddings.projection.bias | 768            
audio_spectrogram_transformer.encoder.layer.0.attention.attention.query.weight | 589,824        
audio_spectrogram_transformer.encoder.layer.0.attention.attention.query.bias | 768            
audio_spectrogram_transformer.encoder.layer.0.attention.attention.key.weight | 589,824        
audio_spectrogram_transformer.encoder.layer.0.attention.attention.key.bias | 768            
audio_spectrogram_transformer.encoder

In [ ]:
from transformers import AutoModelForAudioClassification

model_dir = "MIT/ast-finetuned-audioset-10-10-0.4593"
model = AutoModelForAudioClassification.from_pretrained(model_dir)

# 1. Calculate Total and Trainable Parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters:    {frozen_params:,}")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Total Parameters:     86,594,063
Trainable Parameters: 86,594,063
Frozen Parameters:    0


In [ ]:
print(f"{'Layer Name':<60} | {'Parameters':<15}")
print("-" * 80)
for name, param in model.named_parameters():
    print(f"{name:<60} | {param.numel():<15,}")

Layer Name                                                   | Parameters     
--------------------------------------------------------------------------------
audio_spectrogram_transformer.embeddings.cls_token           | 768            
audio_spectrogram_transformer.embeddings.distillation_token  | 768            
audio_spectrogram_transformer.embeddings.position_embeddings | 932,352        
audio_spectrogram_transformer.embeddings.patch_embeddings.projection.weight | 196,608        
audio_spectrogram_transformer.embeddings.patch_embeddings.projection.bias | 768            
audio_spectrogram_transformer.encoder.layer.0.attention.attention.query.weight | 589,824        
audio_spectrogram_transformer.encoder.layer.0.attention.attention.query.bias | 768            
audio_spectrogram_transformer.encoder.layer.0.attention.attention.key.weight | 589,824        
audio_spectrogram_transformer.encoder.layer.0.attention.attention.key.bias | 768            
audio_spectrogram_transformer.encoder

In [7]:
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForAudioClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. System Setup
# Force PyTorch to utilize efficient CPU multithreading for the matrix math
try:
    torch.set_num_threads(4)
except RuntimeError:
    pass
from datasets import load_from_disk
dataset_path = "/content/drive/MyDrive/Urban8KAudioFiles/processed_test"
test_dataset = load_from_disk(dataset_path)
# 2. Paths
model_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
quantized_model_path = f"{model_dir}/ast_dynamic_int8.pth"

# 3. Reconstruct the Quantized Model Architecture on CPU
print("Loading baseline model structure...")
base_model = AutoModelForAudioClassification.from_pretrained(model_dir)
base_model.to('cpu')

print("Applying dynamic quantization layout...")
model_int8 = torch.quantization.quantize_dynamic(
    base_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Loading saved INT8 weights...")
model_int8.load_state_dict(torch.load(quantized_model_path, map_location='cpu'))
model_int8.eval()

# 4. Evaluation Function
def evaluate_quantized_accuracy(model, dataset):
    """
    Evaluates the quantized model over the provided dataset on the CPU.
    Assumes 'dataset' is a Hugging Face Dataset object containing 'input_values' and 'labels'.
    """
    all_predictions = []
    all_ground_truths = []

    print(f"\nStarting Evaluation over {len(dataset)} samples...")

    with torch.no_grad():
        for i in tqdm(range(len(dataset)), desc="Evaluating"):
            # Extract inputs and add the batch dimension: [1, num_patches, num_mel_bins]
            # Standard AST expects input shape [1, 1211, 128]
            inputs = torch.tensor([dataset[i]["input_values"]]).to('cpu')
            label = dataset[i]["labels"]

            # Forward pass through the INT8 model
            outputs = model(inputs)
            logits = outputs.logits

            # Get the predicted class index
            prediction = torch.argmax(logits, dim=-1).item()

            all_predictions.append(prediction)
            all_ground_truths.append(label)

    # Calculate Statistical Metrics
    accuracy = accuracy_score(all_ground_truths, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_ground_truths,
        all_predictions,
        average='weighted'
    )

    return {
        "accuracy": accuracy,
        "f1_score": f1,
        "precision": precision,
        "recall": recall
    }

# 5. Run Evaluation
# Replace 'test_dataset' with the variable name of your validation/test split
metrics = evaluate_quantized_accuracy(model_int8, test_dataset)

# 6. Display Performance Metrics
print("\n" + "="*40)
print("   QUANTIZED MODEL PERFORMANCE METRICS   ")
print("="*40)
print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"F1-Score:  {metrics['f1_score'] * 100:.2f}%")
print(f"Precision: {metrics['precision'] * 100:.2f}%")
print(f"Recall:    {metrics['recall'] * 100:.2f}%")
print("="*40)

Loading baseline model structure...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Applying dynamic quantization layout...


/tmp/ipykernel_927/3133954185.py:26: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


Loading saved INT8 weights...

Starting Evaluation over 837 samples...


Evaluating: 100%|██████████| 837/837 [59:16<00:00,  4.25s/it]


   QUANTIZED MODEL PERFORMANCE METRICS   
Accuracy:  89.49%
F1-Score:  89.38%
Precision: 90.69%
Recall:    89.49%


# BenchMarking Script

In [9]:
import torch
import time
import numpy as np
from transformers import AutoModelForAudioClassification

# 1. System Multi-threading Standardization
try:
    torch.set_num_threads(4)
except RuntimeError:
    pass

model_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
quantized_model_path = f"{model_dir}/ast_dynamic_int8.pth"

# 2. Extract a Single Sample for the Benchmark
# Replace 'test_dataset' with your active dataset variable name
sample_input = torch.tensor([test_dataset[0]["input_values"]]).to('cpu')
print(f"Benchmark input tensor shape: {sample_input.shape}\n")

# ========================================================
# PHASE 1: Benchmark the Baseline FP32 Model
# ========================================================
print("Loading Baseline FP32 Model...")
base_model = AutoModelForAudioClassification.from_pretrained(model_dir)
base_model.to('cpu')
base_model.eval()

# Warm-up pass (allocates memory layout before timing)
with torch.no_grad():
    _ = base_model(sample_input)

# Benchmark Loop
NUM_RUNS = 50
fp32_latencies = []

print(f"Benchmarking FP32 Model over {NUM_RUNS} iterations...")
with torch.no_grad():
    for _ in range(NUM_RUNS):
        start_time = time.perf_counter()  # High-precision tracker
        _ = base_model(sample_input)
        end_time = time.perf_counter()

        # Convert to milliseconds
        fp32_latencies.append((end_time - start_time) * 1000)

avg_fp32_latency = np.mean(fp32_latencies)

# Clean up memory to avoid cross-contamination
del base_model
import gc; gc.collect()

# ========================================================
# PHASE 2: Benchmark the Quantized INT8 Model
# ========================================================
print("\nLoading and Constructing Quantized INT8 Model...")
base_shell = AutoModelForAudioClassification.from_pretrained(model_dir).to('cpu')
model_int8 = torch.quantization.quantize_dynamic(base_shell, {torch.nn.Linear}, dtype=torch.qint8)
model_int8.load_state_dict(torch.load(quantized_model_path, map_location='cpu'))
model_int8.eval()

# Warm-up pass
with torch.no_grad():
    _ = model_int8(sample_input)

int8_latencies = []

print(f"Benchmarking INT8 Model over {NUM_RUNS} iterations...")
with torch.no_grad():
    for _ in range(NUM_RUNS):
        start_time = time.perf_counter()
        _ = model_int8(sample_input)
        end_time = time.perf_counter()

        int8_latencies.append((end_time - start_time) * 1000)

avg_int8_latency = np.mean(int8_latencies)

# ========================================================
# PHASE 3: Generate System Report
# ========================================================
speedup_factor = avg_fp32_latency / avg_int8_latency

print("\n" + "="*45)
print("       BENCHMARK LATENCY REPORT (CPU)       ")
print("="*45)
print(f"Average FP32 Latency:   {avg_fp32_latency:.2f} ms")
print(f"Average INT8 Latency:   {avg_int8_latency:.2f} ms")
print(f"Latency Reduction:      {((avg_fp32_latency - avg_int8_latency) / avg_fp32_latency) * 100:.2f}%")
print(f"Inference Speedup:      {speedup_factor:.2f} faster")
print("="*45)

Benchmark input tensor shape: torch.Size([1, 1024, 128])

Loading Baseline FP32 Model...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Benchmarking FP32 Model over 50 iterations...

Loading and Constructing Quantized INT8 Model...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

/tmp/ipykernel_927/2540578197.py:57: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(base_shell, {torch.nn.Linear}, dtype=torch.qint8)


Benchmarking INT8 Model over 50 iterations...

       BENCHMARK LATENCY REPORT (CPU)       
Average FP32 Latency:   5475.45 ms
Average INT8 Latency:   4006.38 ms
Latency Reduction:      26.83%
Inference Speedup:      1.37 faster
